# Étude d'ablation — les quatre configurations

Ce notebook **n'exécute aucune simulation** : il lit les artefacts d'une
campagne déjà produite.

```bash
python main.py run --config experiments/ablation.yaml
```

La question à laquelle il répond : *l'écart entre Nearest et BRAM-EV Full vient
de quel composant ?* L'échelle d'ablation n'ajoute qu'un composant à la fois,
sur le **même** monde (même graine, même grille, même flotte, mêmes tirages de
comportement) :

| Configuration | Méthode | Multi-stations | Réputation | Adaptation |
| --- | --- | :-: | :-: | :-: |
| Nearest | `greedy` | non | non | non |
| Multi-Station Only | `multistation` | oui | non | non |
| Multi-Station + Reputation | `multistation_rep` | oui | oui | non |
| BRAM-EV Full | `bramev` | oui | oui | oui |

L'écart entre deux barreaux consécutifs *est* la contribution du composant
ajouté. Le plan est déclaré une seule fois, dans
`src/experiments/methods.py` — les variantes de BRAM-EV sont traitées par
`ablation_variants.ipynb`.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
from IPython.display import Image, display

import src.experiments.methods as methods
from src.pipeline import ablation, figures
from src.pipeline.store import RunStore

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## Choix du run

Une campagne ne porte pas forcément les quatre barreaux :
`latest_with_methods` prend le run le plus récent qui les contient tous, et dit
ce que contiennent les autres s'il n'en trouve aucun. Pour cibler un run précis :
`RunStore.open('../results_grid/<run>')`.

In [2]:
for path in RunStore.list_runs('../results_grid'):
    present = sorted({row['method'] for row in RunStore(path).read_summary()})
    print(f"{path.name}\n    {', '.join(present) or 'aucun cas'}")

20260823T180259Z_seed42_full-grid
    bramev, greedy
20260830T172116Z_seed42_ablation
    bramev, greedy, multistation, multistation_rep
20260831T055117Z_seed42_ablation-variants
    bramev, greedy, multistation, multistation_rep


In [3]:
store = RunStore.latest_with_methods(methods.LADDER, '../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"graine={params.seed} | commit={manifest['git_commit']} | "
      f"cas={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")

../results_grid/20260831T055117Z_seed42_ablation-variants
seed=42 | scénarios=['optimistic', 'balance', 'pessimistic'] | flottes=[50, 100, 150, 200, 250] | méthodes=['greedy', 'multistation', 'multistation_rep', 'bramev'] | 1440 slots (5.0 j) | 40 stations / 4 sociétés | 60 runs
graine=42 | commit=f37d3b1cbc1ae22cd562076e2955d01aee5f3658 | cas=38/60


## Ce qui est réellement activé

Premier contrôle, avant toute lecture de résultat : les drapeaux effectivement
appliqués. Ils voyagent depuis le registre jusqu'à `summary.csv`
(`src/pipeline/tables.py`), donc cette table dit ce que la campagne a fait —
pas ce qu'elle était censée faire.

Un barreau qui ne bascule pas exactement un composant invaliderait toute
l'attribution des gains.

In [4]:
summary = pd.read_csv(store.summary_path)

ORDRE = list(methods.LADDER)
summary['method'] = pd.Categorical(summary['method'], ORDRE + [
    m for m in summary['method'].unique() if m not in ORDRE], ordered=True)

ladder = summary[summary['method'].isin(ORDRE)].copy()

plan = (ladder[['method', 'method_label', 'broadcast', 'reputation',
                'adaptation', 'offer_choice', 'alpha_mode',
                'reputation_scope', 'score_weighting']]
        .drop_duplicates()
        .sort_values('method')
        .set_index('method'))
plan

,method_label,broadcast,reputation,adaptation,offer_choice,alpha_mode,reputation_scope,score_weighting
method,,,,,,,,
greedy,Nearest,False,False,False,utility,sampled,society,duration
multistation,Multi-Station Only,True,False,False,utility,sampled,society,duration
multistation_rep,Multi-Station + Reputation,True,True,False,utility,sampled,society,duration
bramev,BRAM-EV Full,True,True,True,utility,sampled,society,duration


In [5]:
# Un seul composant change d'un barreau au suivant, et les mécanismes internes
# restent ceux de BRAM-EV tout au long : sinon un barreau mélangerait deux effets.
composants = ['broadcast', 'reputation', 'adaptation']
mecanismes = ['offer_choice', 'alpha_mode', 'reputation_scope', 'score_weighting']

for avant, apres, libelle in methods.LADDER_STEPS:
    a, b = plan.loc[avant], plan.loc[apres]
    change = [c for c in composants if a[c] != b[c]]
    derive = [m for m in mecanismes if a[m] != b[m]]
    etat = 'OK' if (len(change) == 1 and not derive) else 'ANOMALIE'
    print(f"{etat:9} {avant:18} -> {apres:18} {libelle:26} "
          f"change={change} mécanismes_dérivés={derive}")

OK        greedy             -> multistation       Recherche multi-stations   change=['broadcast'] mécanismes_dérivés=[]
OK        multistation       -> multistation_rep   Réputation                 change=['reputation'] mécanismes_dérivés=[]
OK        multistation_rep   -> bramev             Adaptation entre stations  change=['adaptation'] mécanismes_dérivés=[]


In [6]:
# Décomposition calculée à la volée depuis summary.csv. Le pipeline persiste
# exactement les mêmes tables (`ablation.csv`, `ablation_mean.csv`) et les
# réécrit à chaque cas ; les recalculer ici rend le notebook utilisable sur une
# campagne encore en cours, interrompue, ou antérieure à l'étude d'ablation.
detail = pd.DataFrame(ablation.detail_rows(summary.to_dict('records')))
moyennes = pd.DataFrame(ablation.mean_rows(detail.to_dict('records')))

print(f"{len(detail)} écarts calculés sur "
      f"{detail[['scenario', 'nb_cars']].drop_duplicates().shape[0]} mondes")

420 écarts calculés sur 10 mondes


## Les quatre barreaux côte à côte

Une ligne par monde (scénario x flotte), une colonne par barreau. C'est la
lecture brute : la progression le long de l'échelle, avant toute soustraction.

In [7]:
METRIQUES = ['exact_satisfaction', 'rate_abs', 'mean_service_rate',
             'slot_waste_rate', 'nb_reservations', 'mean_waiting_time_min',
             'mean_offers_per_demand', 'total_ms_mean']

niveaux = ladder.pivot_table(index=['scenario', 'nb_cars'], columns='method',
                             values=METRIQUES, observed=True)
niveaux

exact_satisfaction                                       mean_offers_per_demand                                      mean_service_rate  \
method                         greedy multistation multistation_rep  bramev                 greedy multistation multistation_rep bramev            greedy   
scenario   nb_cars                                                                                                                                          
balance    50                  0.7596       0.7597           0.7570  0.7563                  0.996        5.973            3.886  3.797            0.0368   
           100                 0.7741       0.7807           0.7807  0.7801                  0.997        6.220            3.959  3.909            0.0764   
           150                 0.7458       0.7814           0.7830  0.7831                  0.982        6.005            3.981  3.974            0.1115   
           200                 0.6992       0.7751           0.7749  0.7750                  0.947        6.040            3.824  3.738            0.1431   
           250                 0.6427       0.7806              NaN     NaN                  0.905        6.062              NaN    NaN            0.1759   
optimistic 50                  0.8789       0.8790           0.8712  0.8714                  0.995        6.155            4.280  4.271            0.0409   
           100                 0.8911       0.8954           0.8906  0.8909                  0.998        5.952            4.534  4.594            0.0820   
           150                 0.8618       0.8931           0.8914  0.8904                  0.987        6.064            4.788  4.775            0.1197   
           200                 0.8121       0.8903           0.8874  0.8864                  0.971        6.126            4.795  4.771            0.1556   
           250                 0.7597       0.8832           0.8840  0.8822                  0.921        6.145            4.857  4.848            0.1901   

                                                         mean_waiting_time_min                                      nb_reservations               \
method             multistation multistation_rep  bramev                greedy multistation multistation_rep bramev          greedy multistation   
scenario   nb_cars                                                                                                                                 
balance    50            0.0368           0.0360  0.0360                 0.012        0.012            0.024  0.000           478.0        478.0   
           100           0.0763           0.0762  0.0762                 2.424        0.264            0.150  0.222           971.0        972.0   
           150           0.1105           0.1113  0.1120                 9.534        0.948            0.624  0.582          1507.0       1459.0   
           200           0.1435           0.1451  0.1441                21.792        1.284            1.596  1.314          2137.0       1965.0   
           250           0.1814              NaN     NaN                37.272        1.962              NaN    NaN          2851.0       2452.0   
optimistic 50            0.0409           0.0402  0.0400                 0.024        0.024            0.012  0.000           442.0        442.0   
           100           0.0817           0.0801  0.0802                 1.692        0.252            0.042  0.024           854.0        850.0   
           150           0.1198           0.1191  0.1191                 7.770        0.396            0.426  0.684          1302.0       1267.0   
           200           0.1574           0.1560  0.1548                17.886        0.930            0.942  1.164          1830.0       1711.0   
           250           0.1946           0.1928  0.1941                28.662        1.842            1.932  2.166          2349.0       2119.0   

                                            rate_abs

In [8]:
# Moyenne sur tous les mondes du run : la vue d'ensemble en une ligne par barreau.
(ladder.groupby('method', observed=True)[METRIQUES]
       .mean()
       .rename(index=methods.label)
       .round(4))

,exact_satisfaction,rate_abs,mean_service_rate,slot_waste_rate,nb_reservations,mean_waiting_time_min,mean_offers_per_demand,total_ms_mean
method,,,,,,,,
Nearest,0.7825,0.1424,0.1132,0.3553,1472.1000,12.7068,0.9699,147.5866
Multi-Station Only,0.8318,0.1429,0.1143,0.3584,1371.5000,0.7914,6.0742,16986.9341
Multi-Station + Reputation,0.8356,0.1372,0.1063,0.3499,1245.2222,0.6387,4.3227,12864.8742
BRAM-EV Full,0.8351,0.1371,0.1063,0.3501,1245.5556,0.6840,4.2974,12804.6841


## Contribution de chaque composant

`ablation_mean.csv` est écrit par le pipeline à chaque campagne
(`src/pipeline/ablation.py`). Chaque ligne est un couple (composant, métrique) :

* `mean_delta` / `mean_delta_pct` — l'écart moyen au barreau précédent ;
* `share_improved` — la **part des mondes** où le composant améliore la
  métrique, sa direction étant déclarée par métrique (moins de no-shows est un
  gain, moins de satisfaction n'en est pas un).

C'est `share_improved` qui compte : un composant qui n'aide que la moitié des
mondes n'a pas de contribution robuste, quelle que soit sa moyenne.

In [9]:
print(ablation.render_mean_table(moyennes.to_dict('records')))

Composant                  Satisfaction exacte   Taux de no-show   Taux de service  Slots réservés perdus  Réservations confirmées
-------------------------  -------------------  ----------------  ----------------  ---------------------  -----------------------

Échelle d'ablation (contribution du composant ajouté)
Recherche multi-stations          +6.8% (100%)       +0.5% (20%)       +0.6% (50%)            +1.0% (10%)              -4.5% (10%)
Réputation                         -0.2% (22%)       -0.4% (67%)       -0.7% (22%)            -0.2% (56%)              -0.5% (44%)
Adaptation entre stations          -0.1% (44%)       -0.1% (44%)       -0.1% (33%)            +0.1% (22%)              +0.0% (33%)

Lecture : écart relatif moyen sur tous les mondes du run (part des mondes où le composant améliore la métrique).


In [10]:
echelle = moyennes[moyennes['kind'] == 'ladder']

contributions = echelle.pivot_table(
    index='component', columns='metric_label',
    values=['mean_delta_pct', 'share_improved'])
contributions.round(3)

mean_delta_pct                                                                                                              \
metric_label              Attente moyenne Besoins satisfaits Demandes rejetées Distance moyenne Latence bout-en-bout No-shows Réservations confirmées   
component                                                                                                                                               
Adaptation entre stations         -13.668             -0.040             2.032            0.523                0.043   -0.022                   0.044   
Recherche multi-stations          -73.639              6.498               NaN            7.022             7427.729   -3.952                  -4.458   
Réputation                         -8.071             -0.051               NaN           18.113                0.493   -0.855                  -0.467   

                                                                                                                                                 \
metric_label              Satisfaction exacte Slots réservés perdus Taux d'occupation Taux de confirmation Taux de no-show Taux de présentation   
component                                                                                                                                         
Adaptation entre stations              -0.057                 0.060            -0.014               -0.144          -0.073               -0.115   
Recherche multi-stations                6.796                 0.999             1.120                2.839           0.523                0.038   
Réputation                             -0.225                -0.215            -0.686               -9.528          -0.374                0.236   

                                                           share_improved                                                                                      \
metric_label              Taux de service Temps de calcul Attente moyenne Besoins satisfaits Demandes rejetées Distance moyenne Latence bout-en-bout No-shows   
component                                                                                                                                                       
Adaptation entre stations          -0.059          -0.084           0.556              0.222             0.222            0.222                0.556    0.222   
Recherche multi-stations            0.562        1869.838           0.800              0.800             0.700            0.100                0.000    0.600   
Réputation                         -0.726           2.360           0.444              0.444             0.000            0.000                0.444    0.556   

                                                                                                                                                    \
metric_label              Réservations confirmées Satisfaction exacte Slots réservés perdus Taux d'occupation Taux de confirmation Taux de no-show   
component                                                                                                                                            
Adaptation entre stations                   0.333               0.444                 0.222             0.556                0.111           0.444   
Recherche multi-stations                    0.100               1.000                 0.100             0.600                0.600           0.200   
Réputation                                  0.444               0.222                 0.556             0.222                0.000           0.667   

                                                                                
metric_label              Taux de présentation Taux de service Temps de calcul  
component                                                                       
Adaptation entre stations                0.111           0.333           0.444  
Recherche multi-stations         

### Les contributions somment-elles à l'écart total ?

Les trois écarts consécutifs doivent reconstituer l'écart Nearest -> BRAM-EV
Full, monde par monde. Un résidu non nul signalerait un barreau manquant ou un
`summary.csv` incohérent.

In [11]:
pas = detail[(detail['kind'] == 'ladder') &
             (detail['metric'] == 'exact_satisfaction')]

somme = (pas.groupby(['scenario', 'nb_cars'])['delta'].sum()
            .rename('somme_des_contributions'))

extremes = ladder.pivot_table(index=['scenario', 'nb_cars'], columns='method',
                              values='exact_satisfaction', observed=True)
total = (extremes[methods.LADDER[-1]] - extremes[methods.LADDER[0]]
         ).rename('ecart_total')

controle = pd.concat([somme, total], axis=1)
controle['residu'] = (controle['somme_des_contributions']
                      - controle['ecart_total']).abs()
print(f"résidu maximal : {controle['residu'].max():.2e}")
controle.round(6)

résidu maximal : 8.33e-17


somme_des_contributions  ecart_total  residu
scenario   nb_cars                                              
balance    50                       -0.0033      -0.0033     0.0
           100                       0.0060       0.0060     0.0
           150                       0.0373       0.0373     0.0
           200                       0.0758       0.0758     0.0
           250                       0.1379          NaN     NaN
optimistic 50                       -0.0075      -0.0075     0.0
           100                      -0.0002      -0.0002     0.0
           150                       0.0286       0.0286     0.0
           200                       0.0743       0.0743     0.0
           250                       0.1225       0.1225     0.0

## Dispersion : la moyenne cache quoi ?

Un écart moyen positif peut n'être porté que par un monde. Voici la
distribution des contributions par composant, sur toutes les tailles de flotte
et tous les scénarios.

In [12]:
for metrique in ['exact_satisfaction', 'rate_abs', 'mean_service_rate']:
    bloc = detail[(detail['kind'] == 'ladder') & (detail['metric'] == metrique)]
    if bloc.empty:
        continue
    libelle = bloc['metric_label'].iloc[0]
    print(f"\n=== {libelle} — écart au barreau précédent ===")
    stats = (bloc.groupby('component')['delta']
                 .agg(['count', 'min', 'median', 'mean', 'max'])
                 .round(6))
    stats['mondes_améliorés'] = bloc.groupby('component')['improvement'].mean().round(3)
    display(stats)


=== Satisfaction exacte — écart au barreau précédent ===


,count,min,median,mean,max,mondes_améliorés
component,,,,,,
Adaptation entre stations,9,-0.0018,-0.00060,-0.000489,0.0003,0.444
Recherche multi-stations,10,0.0001,0.03345,0.049350,0.1379,1.000
Réputation,9,-0.0078,-0.00170,-0.001967,0.0016,0.222



=== Taux de no-show — écart au barreau précédent ===


,count,min,median,mean,max,mondes_améliorés
component,,,,,,
Adaptation entre stations,9,-0.0008,0.0000,-0.000078,0.0013,0.444
Recherche multi-stations,10,-0.0019,0.0003,0.000530,0.0030,0.200
Réputation,9,-0.0018,-0.0003,-0.000222,0.0016,0.667



=== Taux de service — écart au barreau précédent ===


,count,min,median,mean,max,mondes_améliorés
component,,,,,,
Adaptation entre stations,9,-0.0012,0.00000,-0.000033,0.0013,0.333
Recherche multi-stations,10,-0.0010,0.00005,0.001090,0.0055,0.500
Réputation,9,-0.0018,-0.00070,-0.000522,0.0016,0.222


In [13]:
# Le composant aide-t-il davantage quand la ressource devient rare ?
# Une contribution qui croît avec la flotte est une contribution qui porte
# sur la congestion, non sur le hasard du tirage.
(detail[(detail['kind'] == 'ladder') &
        (detail['metric'] == 'exact_satisfaction')]
 .pivot_table(index='component', columns='nb_cars', values='delta_pct')
 .round(2))

nb_cars,50,100,150,200,250
component,,,,,
Adaptation entre stations,-0.03,-0.02,-0.05,-0.05,-0.20
Recherche multi-stations,0.01,0.67,4.20,10.24,18.86
Réputation,-0.62,-0.27,0.01,-0.18,0.09


## Figures

Déjà écrites par le pipeline dans `figures/`. Les régénérer après avoir modifié
`src/pipeline/figures.py` :

```bash
python main.py report --latest
```

In [14]:
for nom in ['ablation_components', 'scalability']:
    chemin = store.figure_path(nom)
    if chemin.is_file():
        print(chemin.name)
        display(Image(filename=str(chemin)))

for chemin in sorted(store.figures_dir.glob('ablation_ladder_*.png')):
    print(chemin.name)
    display(Image(filename=str(chemin)))

In [15]:
# Ou reconstruire une figure en mémoire, sans repasser par le disque.
figures.fig_ablation_components(summary.to_dict('records'))

<Figure size 1510x440 with 4 Axes>

## Garde-fous de lecture

Deux composants ne peuvent structurellement rien montrer sur une campagne trop
courte ou trop peu contrainte. Un `+0.0%` doit alors se lire « mécanisme jamais
sollicité », pas « composant inutile ».

1. **L'adaptation entre stations** ne se déclenche que tous les
   `SOCIETY_UPDATE_INTERVAL` slots (144 par défaut, 12 h).
2. **La recherche multi-stations** n'a d'effet que si la requête atteint
   effectivement plus d'une station, et que la capacité est contrainte : avec
   des bornes libres partout, toute demande est servie de toute façon.

In [16]:
config = next(store.iter_results())['config']
intervalle = config['society_update_interval']
mises_a_jour = params.total_time // intervalle

print(f"horizon = {params.total_time} slots | "
      f"intervalle d'adaptation = {intervalle} slots")
print(f"-> l'apprentissage collectif se déclenche {mises_a_jour} fois")
if mises_a_jour == 0:
    print("   ATTENTION : jamais déclenché — la contribution mesurée de "
          "l'adaptation est nulle par construction.")

horizon = 1440 slots | intervalle d'adaptation = 144 slots
-> l'apprentissage collectif se déclenche 10 fois


In [17]:
diffusion = (ladder.groupby('method', observed=True)
                   .agg(offres_par_demande=('mean_offers_per_demand', 'mean'),
                        taux_de_service=('mean_service_rate', 'mean'),
                        demandes_rejetees=('nb_rejected_request', 'mean'))
                   .round(3))
print(diffusion, end='\n\n')

gain_offres = (diffusion.loc['multistation', 'offres_par_demande']
               - diffusion.loc['greedy', 'offres_par_demande'])
if gain_offres <= 0:
    print("ATTENTION : la diffusion ne produit pas plus d'offres par demande. "
          "Le rayon de recherche est trop petit devant l'espacement des "
          "stations : le premier barreau n'est pas testé.")
else:
    print(f"La diffusion apporte {gain_offres:+.2f} offre(s) par demande.")

                  offres_par_demande  taux_de_service  demandes_rejetees
method                                                                  
greedy                         0.970            0.113             67.000
multistation                   6.074            0.114              8.200
multistation_rep               4.323            0.106           2257.556
bramev                         4.297            0.106           2321.667

La diffusion apporte +5.10 offre(s) par demande.


## Santé du run

Un invariant violé ou un diagnostic massif rend toute conclusion suspecte :
à vérifier avant de citer un chiffre.

In [18]:
print('invariants OK :', bool(summary['invariant_ok'].all()))
print('réservations non résolues :', int(summary['nb_unresolved'].sum()))
print('pannes :', int(summary['nb_breakdowns'].sum()))

vus = set()
for resultat in store.iter_results():
    for message in resultat['behaviors'].get('diagnostics', []):
        if message not in vus:
            vus.add(message)
            print(f"\n[diagnostic] {message}")

invariants OK : True
réservations non résolues : 48
pannes : 284

[diagnostic] Aucune annulation anticipée réalisée alors que le scénario en prévoit : toutes sont reclassées en tardives. Une annulation ne peut être anticipée que si la réservation est prise suffisamment à l'avance ; avec un délai requête → arrivée quasi nul, la distinction anticipé / tardif n'est pas mesurable et la probabilité 'early' du scénario se réalise en 'late'. Voir 'reclassified' et 'median_lead_slots'.

[diagnostic] Délai moyen requête → arrivée prévue < 1 slot : les stations proposent des créneaux immédiats (rayon de recherche petit devant la vitesse par slot). Aucune réservation n'est prise à l'avance.
